# Example processing script: Heart rate and acceleration

## Setup

I have switched from `conda` to `uv` (it's much faster and simpler to configure). See installation here: [uv installation](https://docs.astral.sh/uv/getting-started/installation/).

- On Macos, if you use [homebrew](https://brew.sh/), install with: `brew install uv`
- Next easiest option is: `pip install uv`

After installing `uv`, navigate to `BEATmonitor/src/analysis/` and run: `uv init`

In [45]:
import os, sys # Operating system interaction
import re # Text search tools
import pytz # Timezone tools
import pandas as pd # Data handling
import holoviews as hv # Visualisation
import hvplot.pandas
import neurokit2 as nk # PPG processing
# Import beatwatch processing tools
from beatwatch_process.parsers import Parser

TIMEZONE = "America/Toronto" # Set timezone of data collection
DATA_PATH = "example_data/" # Path to raw data
OUTPUT_PATH = "example_results/" # Path to store results
SUBFOLDERS = ["", "figures/", "processed/"]

MAX_GAP_HR = round(3.9 * 40) # Gaps larger than 3 samples at ~40 ms
MAX_GAP_ACCEL = round(3.9 * 80) # Gaps larger than 3 samples at ~80 ms

RATE_UPSAMPLE = 1000
RATE_DOWNSAMPLE = 40 # 25 Hz

tz = pytz.timezone(TIMEZONE) # Initialize timezone
parser = Parser(TIMEZONE) # Create parser object

for f in SUBFOLDERS:
    os.makedirs(OUTPUT_PATH + f, exist_ok=True) # Create folder if needed

print(f" - Executable: {sys.executable}")
print(f" - Version: {sys.version}")
print(f" - Working directory: {os.getcwd()}")

 - Executable: /Users/mayaflannery/Desktop/projects/BEATmonitor/src/analysis/.venv/bin/python
 - Version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 11:23:37) [Clang 14.0.6 ]
 - Working directory: /Users/mayaflannery/Desktop/projects/BEATmonitor/src/analysis


## Read data

### Search for valid files

In [2]:
def get_valid_watch_files(data_directory):
    match_data = re.compile(r".*(time).*\.(csv|hr|sv)$", re.IGNORECASE)
    file_list = [f for f in os.listdir(data_directory) if match_data.fullmatch(f)]
    return file_list

f_data = get_valid_watch_files(DATA_PATH)

[print(f) for f in f_data]

10-09_time_14-38-58_2cee_W010.csv
10-09_time_14-38-58_0510_W002.csv
10-09_time_14-38-58_7eed_W004.csv


[None, None, None]

### Parse files

In [3]:
data = {}

for f in f_data:
    data[f] = parser.parse_file(DATA_PATH + f)
    data[f]['data_hr'].set_index('time_absolute', inplace=True)
    data[f]['data_accel'].set_index('time_absolute', inplace=True)

Reading example_data/10-09_time_14-38-58_2cee_W010.csv
Reading example_data/10-09_time_14-38-58_0510_W002.csv
Reading example_data/10-09_time_14-38-58_7eed_W004.csv


### Parsed result

`data` is a dictionary of filenames containing metadata and dataframes for each data type:

To iterate through data, use `data.items()`

In [4]:
for k, v in data.items():
    print(f'Key: {k}')
    for i in v:
        print(f' item: {i}')

Key: 10-09_time_14-38-58_2cee_W010.csv
 item: metadata
 item: data_hr
 item: data_accel
Key: 10-09_time_14-38-58_0510_W002.csv
 item: metadata
 item: data_hr
 item: data_accel
Key: 10-09_time_14-38-58_7eed_W004.csv
 item: metadata
 item: data_hr
 item: data_accel


### Summary of parsed data

In [5]:
def print_metadata(data):
    record = []
    for i in data:
        data[i]['metadata']['file_name'] = i
        record.append(data[i]['metadata'])
    return pd.DataFrame.from_records(record).set_index("file_name")

metadata = print_metadata(data)
metadata.to_csv(OUTPUT_PATH + "parsed_metadata.csv")

metadata

,Parsed_on,StudyName,StudyInstance,Name,Serial,MAC,PhysicalID,start_State,start_DateTime,start_UNIXTimeStamp,...,stop_DateTime,stop_UNIXTimeStamp,stop_BatteryLife,stop_FreeStorage,stop_SamplesWritten,n_samples_hr,n_samples_accel,n_survey_responses,duration_hr,duration_accel
file_name,,,,,,,,,,,,,,,,,,,,,
10-09_time_14-38-58_2cee_W010.csv,2025-11-24T13:42:29.981821+00:00,NA,NA,10-09T14:38:58_2cee_W010,7440756e-c4ef9af8,e6:3d:7e:d8:2c:ee,W010,START_RECORD,Thu Oct 9 2025 09:38:58 GMT-0500,2025-10-09T14:38:58.714Z,...,Thu Oct 9 2025 10:33:49 GMT-0500,2025-10-09T15:33:49.455Z,68,5436860,"{'hrm': 70440, 'accel': 36762}",70440,36762,0,0 days 00:54:50.734000,0 days 00:54:50.680000
10-09_time_14-38-58_0510_W002.csv,2025-11-24T13:42:30.189334+00:00,NA,NA,10-09T14:38:58_0510_W002,6cc37388-a01c03b0,ee:22:5c:2e:05:10,W002,START_RECORD,Thu Oct 9 2025 10:38:58 GMT-0400,2025-10-09T14:38:58.215Z,...,Thu Oct 9 2025 11:33:47 GMT-0400,2025-10-09T15:33:47.748Z,72,20912,"{'hrm': 69288, 'accel': 37303}",69288,37303,0,0 days 00:54:49.526000,0 days 00:54:49.519000
10-09_time_14-38-58_7eed_W004.csv,2025-11-24T13:42:30.311073+00:00,NA,NA,10-09T14:38:58_7eed_W004,d408d083-1d653f7c,d6:a5:0f:e3:7e:ed,W004,START_RECORD,Thu Oct 9 2025 10:38:58 GMT-0400,2025-10-09T14:38:58.508Z,...,Thu Oct 9 2025 11:33:48 GMT-0400,2025-10-09T15:33:48.329Z,73,5354940,"{'hrm': 69855, 'accel': 36581}",69855,36581,0,0 days 00:54:49.812000,0 days 00:54:49.795000


### Structure of parsed dataframes

#### Heart rate data

In [6]:
data["10-09_time_14-38-58_2cee_W010.csv"]["data_hr"].head()

,time_elapsed,heart_rate_bpm,confidence,ppg_raw,ppg_filter
time_absolute,,,,,
2025-10-09 10:39:13.186000-04:00,0 days 00:00:14.472000,77,81,6486,6144
2025-10-09 10:39:13.235000-04:00,0 days 00:00:14.521000,77,81,6462,-768
2025-10-09 10:39:13.309000-04:00,0 days 00:00:14.595000,77,81,6462,-256
2025-10-09 10:39:13.350000-04:00,0 days 00:00:14.636000,77,81,6478,4096
2025-10-09 10:39:13.396000-04:00,0 days 00:00:14.682000,77,81,6462,-512


#### Acceleration data

In [7]:
data["10-09_time_14-38-58_2cee_W010.csv"]["data_accel"].head()

,time_elapsed,x,y,z,magnitude,difference
time_absolute,,,,,,
2025-10-09 10:38:58.827000-04:00,0 days 00:00:00.113000,-60,-22,-984,986,3
2025-10-09 10:38:58.912000-04:00,0 days 00:00:00.198000,-59,-21,-988,990,4
2025-10-09 10:38:58.986000-04:00,0 days 00:00:00.272000,-61,-19,-991,994,4
2025-10-09 10:38:59.066000-04:00,0 days 00:00:00.352000,-63,-21,-994,996,3
2025-10-09 10:38:59.146000-04:00,0 days 00:00:00.432000,-66,-22,-997,1000,5


### IMPORTANT: Uneven sample rate

#### Compute difference between samples

In [8]:
for k, v in data.items():
    v["data_hr"]["diff"] = v["data_hr"].index.diff().total_seconds() * 1000
    v["data_accel"]["diff"] = v["data_accel"].index.diff().total_seconds() * 1000

#### Label large gaps in data

In [9]:
for k, v in data.items():
    v["data_hr"]["gap"] = v["data_hr"]["diff"] >= MAX_GAP_HR
    v["data_accel"]["gap"] = v["data_accel"]["diff"] >= MAX_GAP_ACCEL

# New df:
data["10-09_time_14-38-58_2cee_W010.csv"]["data_accel"].head()

,time_elapsed,x,y,z,magnitude,difference,diff,gap
time_absolute,,,,,,,,
2025-10-09 10:38:58.827000-04:00,0 days 00:00:00.113000,-60,-22,-984,986,3,NaN,False
2025-10-09 10:38:58.912000-04:00,0 days 00:00:00.198000,-59,-21,-988,990,4,85.0,False
2025-10-09 10:38:58.986000-04:00,0 days 00:00:00.272000,-61,-19,-991,994,4,74.0,False
2025-10-09 10:38:59.066000-04:00,0 days 00:00:00.352000,-63,-21,-994,996,3,80.0,False
2025-10-09 10:38:59.146000-04:00,0 days 00:00:00.432000,-66,-22,-997,1000,5,80.0,False


Visualize the distribution of sample rates:

In [39]:
subplots = []
for k, v in data.items(): 
    s = v["data_hr"].loc[v["data_hr"]["gap"] == False, "diff"]
    plt = s.hvplot.hist(bins=round(s.max() - s.min())*10)
    subplots.append(plt)
hv.Layout(subplots).cols(1)

:Layout
   .Histogram.I   :Histogram   [diff]   (Count)
   .Histogram.II  :Histogram   [diff]   (Count)
   .Histogram.III :Histogram   [diff]   (Count)

Most values are approximately 40ms, some 'missed' samples are ~80ms / ~120ms

## Preprocess

### Upsample to 1000 Hz

In [49]:
def upsample(df, max_gap=150):
    # Create new time index
    time = pd.date_range(
        start=df.index.min(), 
        end=df.index.max(),
        freq=pd.to_timedelta(RATE_UPSAMPLE * 1000)
    )
    df_out = df.copy().reindex(time)
    # Do not interpolate above max_gap
    df_out[df_out.select_dtypes(include="number").columns] = (
        df_out.select_dtypes(include="number").interpolate(limit=max_gap)
    )
    df_out["gap"] = df_out["gap"].astype("boolean").ffill()
    return df_out

for k, v in data.items():
    v["data_hr"] = upsample(v["data_hr"], MAX_GAP_HR)
    v["data_accel"] = upsample(v["data_accel"], MAX_GAP_ACCEL)

#### New data structure

In [50]:
data["10-09_time_14-38-58_2cee_W010.csv"]["data_hr"].head()

,time_elapsed,heart_rate_bpm,confidence,ppg_raw,ppg_filter,diff,gap
2025-10-09 10:39:13.186000-04:00,0 days 00:00:14.472000,77.0,81.0,6486.000000,6144.000000,NaN,False
2025-10-09 10:39:13.187000-04:00,0 days 00:00:14.473000,77.0,81.0,6485.510204,6002.938776,NaN,False
2025-10-09 10:39:13.188000-04:00,0 days 00:00:14.474000,77.0,81.0,6485.020408,5861.877551,NaN,False
2025-10-09 10:39:13.189000-04:00,0 days 00:00:14.475000,77.0,81.0,6484.530612,5720.816327,NaN,False
2025-10-09 10:39:13.190000-04:00,0 days 00:00:14.476000,77.0,81.0,6484.040816,5579.755102,NaN,False


In [51]:
data["10-09_time_14-38-58_2cee_W010.csv"]["data_accel"].head()

,time_elapsed,x,y,z,magnitude,difference,diff,gap
2025-10-09 10:38:58.827000-04:00,0 days 00:00:00.113000,-60.000000,-22.000000,-984.000000,986.000000,3.000000,NaN,False
2025-10-09 10:38:58.828000-04:00,0 days 00:00:00.114000,-59.988235,-21.988235,-984.047059,986.047059,3.011765,NaN,False
2025-10-09 10:38:58.829000-04:00,0 days 00:00:00.115000,-59.976471,-21.976471,-984.094118,986.094118,3.023529,NaN,False
2025-10-09 10:38:58.830000-04:00,0 days 00:00:00.116000,-59.964706,-21.964706,-984.141176,986.141176,3.035294,NaN,False
2025-10-09 10:38:58.831000-04:00,0 days 00:00:00.117000,-59.952941,-21.952941,-984.188235,986.188235,3.047059,NaN,False


#### Visualization

In [53]:
subplots = []
for k, v in data.items():
    trace = v["data_accel"].iloc[::RATE_DOWNSAMPLE]["difference"].hvplot.scatter()
    subplots.append(trace)
hv.Layout(subplots).cols(1)

:Layout
   .Scatter.Difference.I   :Scatter   [index]   (difference)
   .Scatter.Difference.II  :Scatter   [index]   (difference)
   .Scatter.Difference.III :Scatter   [index]   (difference)

### Downsampling

Example: subset of acceleration data

In [72]:
df_sub = data['10-09_time_14-38-58_2cee_W010.csv']['data_accel']
df_sub = df_sub.iloc[10000:20000]

hv.Overlay([
    df_sub['x'].hvplot.line().opts(color='black'), # Upsampled points
    df_sub.loc[::RATE_DOWNSAMPLE]['x'].hvplot.scatter().opts(color='red'), # downsampled evenly
    df_sub.loc[::RATE_DOWNSAMPLE]['x'].hvplot.line().opts(color='red', alpha=0.5) # downsampled evenly
])

:Overlay
   .Curve.X.I  :Curve   [index]   (x)
   .Scatter.X  :Scatter   [index]   (x)
   .Curve.X.II :Curve   [index]   (x)

## Write processed data

Write all processed data to files (separate file per watch/dataset)

In [73]:
for k, v in data.items():
    v["data_hr"].to_parquet(OUTPUT_PATH + f"processed/heart_rate/{k.strip(".csv")}_1KHz.parquet")
    v["data_accel"].to_parquet(OUTPUT_PATH + f"processed/acceleration/{k.strip(".csv")}_1KHz.parquet")

### Optional

#### Drop/Downsample

Example: `time_elapsed` and `diff` columns can be dropped; sample frequency reduced to 40ms

- file size is reduced from 116MB --> 3MB

In [78]:
for k, v in data.items():
    (
        v["data_hr"][::RATE_DOWNSAMPLE]  # Filter every RATE_DOWNSAMPLE-th sample
        .drop(columns=["time_elapsed", "diff"])  # Drop unused columns
        .to_parquet(
            OUTPUT_PATH + f"processed/heart_rate/{k.strip('.csv')}_40Hz.parquet"
        )
    )
    (
        v["data_accel"][::RATE_DOWNSAMPLE]  # Filter samples
        .drop(columns=["time_elapsed", "diff"])  # Drop columns
        .to_parquet(
            OUTPUT_PATH + f"processed/acceleration/{k.strip('.csv')}_40Hz.parquet"
        )
    )

#### Single data file

Add watch name to data frame and save single file for each data type (hr, acceleration)

In [81]:
data_accel = pd.DataFrame()
data_hr = pd.DataFrame()

for k, v in data.items():
    # Get watch name
    watch_name = re.search(r"(W.*)\..*$", f).group(1)
    # Add name to hr dataframe
    v["data_hr"]["watch"] = watch_name
    v["data_hr"]["watch"] = v["data_hr"]["watch"].astype("category")
    # Add name to accel dataframe
    v["data_accel"]["watch"] = watch_name
    v["data_accel"]["watch"] = v["data_accel"]["watch"].astype("category")
    # Create single hr, accel dataframes
    data_hr = pd.concat(
        [
            data_hr,
            v["data_hr"][::RATE_DOWNSAMPLE].drop(columns=["time_elapsed", "diff"]),
        ]
    )
    data_Accel = pd.concat(
        [
            data_accel,
            v["data_accel"][::RATE_DOWNSAMPLE].drop(columns=["time_elapsed", "diff"]),
        ]
    )



<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 9869653 entries, 2025-10-09 10:38:58.827000-04:00 to 2025-10-09 11:33:48.303000-04:00
Data columns (total 9 columns):
 #   Column        Dtype          
---  ------        -----          
 0   time_elapsed  timedelta64[ns]
 1   x             float64        
 2   y             float64        
 3   z             float64        
 4   magnitude     float64        
 5   difference    float64        
 6   diff          float64        
 7   gap           boolean        
 8   watch         object         
dtypes: boolean(1), float64(6), object(1), timedelta64[ns](1)
memory usage: 696.5+ MB
